<a href="https://colab.research.google.com/github/jusegora/legaltech-contract-analyzer/blob/main/analytics/BigData_Judge_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

# Inicializar la sesión de Spark
spark = SparkSession.builder \
    .appName("LegalTech_BigData_Analytics") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("✅ Sesión de PySpark iniciada correctamente.")
print(f"Versión de Spark: {spark.version}")

✅ Sesión de PySpark iniciada correctamente.
Versión de Spark: 4.0.5


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

print("⏳ Generando 100,000 registros judiciales sintéticos...")

# Crear un DataFrame base con 100,000 filas (usando spark.range para máximo rendimiento)
df = spark.range(100000)

# Añadir columnas simulando datos legales usando funciones de PySpark
df_legal = df.withColumn("case_id", F.concat(F.lit("CASE-"), F.col("id"))) \
    .withColumn("case_type", F.when(F.rand() < 0.4, "Laboral")
                                .when(F.rand() < 0.7, "Civil")
                                .when(F.rand() < 0.9, "Penal")
                                .otherwise("Administrativo")) \
    .withColumn("judge_id", F.concat(F.lit("JUEZ-"), (F.rand() * 50).cast("int"))) \
    .withColumn("verdict", F.when(F.rand() < 0.5, "Favorable").otherwise("Desfavorable")) \
    .withColumn("amount_claimed", (F.rand() * 50000 + 1000).cast("int")) \
    .withColumn("year", (F.rand() * 10 + 2015).cast("int")) \
    .drop("id")

# Mostrar los primeros 5 registros
print("✅ Dataset generado con éxito. Aquí hay una muestra:")
df_legal.show(5)

# Contar cuántos registros hay en total (demostrando la escala)
total_records = df_legal.count()
print(f"📊 Total de registros en el Big Data: {total_records}")

⏳ Generando 100,000 registros judiciales sintéticos...
✅ Dataset generado con éxito. Aquí hay una muestra:
+-------+---------+--------+------------+--------------+----+
|case_id|case_type|judge_id|     verdict|amount_claimed|year|
+-------+---------+--------+------------+--------------+----+
| CASE-0|  Laboral| JUEZ-12|Desfavorable|         23933|2015|
| CASE-1|    Civil| JUEZ-21|   Favorable|          4017|2022|
| CASE-2|  Laboral| JUEZ-43|   Favorable|         27585|2021|
| CASE-3|  Laboral| JUEZ-30|   Favorable|          6946|2021|
| CASE-4|    Civil| JUEZ-30|Desfavorable|         35845|2024|
+-------+---------+--------+------------+--------------+----+
only showing top 5 rows
📊 Total de registros en el Big Data: 100000


In [4]:
print("🔍 Analizando el sesgo de los jueces en casos Laborales...")

# Filtrar solo casos laborales
labor_cases = df_legal.filter(F.col("case_type") == "Laboral")

# Agrupar por juez y calcular el total de casos y el porcentaje de fallos favorables
judge_stats = labor_cases.groupBy("judge_id") \
    .agg(
        F.count("case_id").alias("total_casos"),
        F.sum(F.when(F.col("verdict") == "Favorable", 1).otherwise(0)).alias("fallos_favorables")
    ) \
    .withColumn("porcentaje_favorable", F.round((F.col("fallos_favorables") / F.col("total_casos")) * 100, 2)) \
    .orderBy(F.desc("porcentaje_favorable"))

# Mostrar los 10 jueces más "pro-trabajador"
print("⚖️ Top 10 Jueces con mayor porcentaje de fallos favorables en casos Laborales:")
judge_stats.show(10)

🔍 Analizando el sesgo de los jueces en casos Laborales...
⚖️ Top 10 Jueces con mayor porcentaje de fallos favorables en casos Laborales:
+--------+-----------+-----------------+--------------------+
|judge_id|total_casos|fallos_favorables|porcentaje_favorable|
+--------+-----------+-----------------+--------------------+
| JUEZ-27|        806|              424|               52.61|
| JUEZ-45|        784|              410|                52.3|
| JUEZ-43|        787|              411|               52.22|
| JUEZ-23|        768|              401|               52.21|
| JUEZ-28|        827|              431|               52.12|
| JUEZ-21|        775|              403|                52.0|
| JUEZ-30|        782|              406|               51.92|
| JUEZ-22|        821|              426|               51.89|
| JUEZ-48|        779|              402|                51.6|
| JUEZ-44|        804|              414|               51.49|
+--------+-----------+-----------------+-----------------

In [5]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

print("🧠 Entrenando modelo predictivo (Machine Learning en Big Data)...")

# 1. Indexar variables categóricas (Spark ML solo acepta números)
indexer_case = StringIndexer(inputCol="case_type", outputCol="case_type_idx", handleInvalid="keep")
indexer_judge = StringIndexer(inputCol="judge_id", outputCol="judge_idx", handleInvalid="keep")
indexer_label = StringIndexer(inputCol="verdict", outputCol="label", handleInvalid="keep")

# 2. Ensamblar las características (Features) en un solo vector
assembler = VectorAssembler(
    inputCols=["case_type_idx", "judge_idx", "amount_claimed", "year"],
    outputCol="features"
)

# 3. Definir el algoritmo (Regresión Logística)
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# 4. Crear el Pipeline de nivel producción
pipeline = Pipeline(stages=[indexer_case, indexer_judge, indexer_label, assembler, lr])

# 5. Dividir los datos: 80% para entrenar, 20% para probar
train_data, test_data = df_legal.randomSplit([0.8, 0.2], seed=42)

# 6. Entrenar el modelo (Aquí es donde Spark brilla)
model = pipeline.fit(train_data)

# 7. Hacer predicciones sobre el 20% de datos que el modelo nunca vio
predictions = model.transform(test_data)

# 8. Evaluar la precisión (Métrica AUC-ROC)
evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)

print(f"✅ Modelo entrenado exitosamente.")
print(f"📊 Métrica AUC-ROC en el conjunto de prueba: {auc:.4f}")
print("\n🔍 Muestra de predicciones (Veredicto Real vs Predicción):")
predictions.select("case_type", "judge_id", "amount_claimed", "verdict", "prediction").show(5)

🧠 Entrenando modelo predictivo (Machine Learning en Big Data)...


IllegalArgumentException: requirement failed: rawPredictionCol vectors must have length=2, but got 3

In [6]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

print("🧠 Entrenando modelo predictivo (Machine Learning en Big Data)...")

# 1. Indexar variables categóricas
indexer_case = StringIndexer(inputCol="case_type", outputCol="case_type_idx", handleInvalid="keep")
indexer_judge = StringIndexer(inputCol="judge_id", outputCol="judge_idx", handleInvalid="keep")
# CORRECCIÓN: Usamos "skip" en el label para asegurar que solo haya 2 clases (0 y 1)
indexer_label = StringIndexer(inputCol="verdict", outputCol="label", handleInvalid="skip")

# 2. Ensamblar las características (Features)
assembler = VectorAssembler(
    inputCols=["case_type_idx", "judge_idx", "amount_claimed", "year"],
    outputCol="features"
)

# 3. Definir el algoritmo
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# 4. Crear el Pipeline
pipeline = Pipeline(stages=[indexer_case, indexer_judge, indexer_label, assembler, lr])

# 5. Dividir los datos: 80% entrenamiento, 20% prueba
train_data, test_data = df_legal.randomSplit([0.8, 0.2], seed=42)

# 6. Entrenar el modelo
model = pipeline.fit(train_data)

# 7. Hacer predicciones
predictions = model.transform(test_data)

# 8. Evaluar la precisión usando el evaluador Multiclase (más robusto)
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

acc = evaluator_acc.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

print(f"✅ Modelo entrenado exitosamente.")
print(f"📊 Precisión (Accuracy): {acc:.4f}")
print(f"📊 F1-Score: {f1:.4f}")
print("\n🔍 Muestra de predicciones (Veredicto Real vs Predicción):")
predictions.select("case_type", "judge_id", "amount_claimed", "verdict", "prediction").show(5)

🧠 Entrenando modelo predictivo (Machine Learning en Big Data)...
✅ Modelo entrenado exitosamente.
📊 Precisión (Accuracy): 0.5094
📊 F1-Score: 0.5091

🔍 Muestra de predicciones (Veredicto Real vs Predicción):
+---------+--------+--------------+---------+----------+
|case_type|judge_id|amount_claimed|  verdict|prediction|
+---------+--------+--------------+---------+----------+
|    Civil| JUEZ-27|          3561|Favorable|       1.0|
|    Penal| JUEZ-37|         24642|Favorable|       1.0|
|  Laboral| JUEZ-23|         21382|Favorable|       0.0|
|  Laboral| JUEZ-11|         28943|Favorable|       1.0|
|    Penal|  JUEZ-6|         22728|Favorable|       1.0|
+---------+--------+--------------+---------+----------+
only showing top 5 rows
